# Cognitive LLM — Phase 1: Ablation Comparison Notebook

Compare different block configurations on GSM8K to measure each block's contribution.

Each experiment:
1. Loads a fresh LoRA + cognitive block config
2. Trains 100 optimizer steps on GSM8K
3. Records loss curve and GSM8K accuracy
4. Plots all configs side by side

In [1]:
# Install dependencies
%pip install -q transformers datasets accelerate bitsandbytes peft trl wandb lm-eval pyyaml matplotlib


In [2]:
# Clone/update repo and ensure latest code is loaded
from pathlib import Path
import os
import subprocess
import sys

repo_dir = Path('/content/cognitive-llm')

# Always force a fresh clone to avoid stale code
if repo_dir.exists():
    import shutil
    shutil.rmtree(repo_dir)

subprocess.run(
    ['git', 'clone', 'https://github.com/RiyadMehdi7/cognitive-llm.git', str(repo_dir)],
    check=True,
)
os.chdir(repo_dir)

print(f'Working directory: {Path.cwd()}')
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Clear any cached modules
for name in list(sys.modules):
    if name.startswith('cognitive_llm'):
        del sys.modules[name]

print('Ready — using latest code from GitHub')

Working directory: /content/cognitive-llm
Ready — using latest code from GitHub


In [3]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), 'GPU runtime required'
print(f'cwd: {os.getcwd()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


cwd: /content/cognitive-llm
GPU: Tesla T4
Memory: 15.6 GB


In [ ]:
# Load SmolLM3 3B with 4-bit quantization for Colab T4
model_id = 'HuggingFaceTB/SmolLM3-3B'

capability = torch.cuda.get_device_capability(0)
use_bf16 = capability[0] >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f'Using compute dtype: {compute_dtype}')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Quick load to verify config, then immediately free GPU memory
_test_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    dtype=compute_dtype,
)
print(f'Model verified: {_test_model.config.hidden_size}d, {_test_model.config.num_hidden_layers} layers')
del _test_model
import gc
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory freed. Available: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')

In [ ]:
# ── Experiment runner ──
# Builds model, trains, evaluates, returns results dict.
# Re-run this cell then call run_experiment() with different configs.

import gc, copy, re
import matplotlib.pyplot as plt
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from cognitive_llm.models.cognitive_model import CognitiveModel
from cognitive_llm.training.trainer import CognitiveTrainer
from datasets import load_dataset

# Store all experiment results for comparison
all_results = {}

TRAINING_CONFIG = {
    'learning_rate': 5e-5,
    'max_steps': 800,          # 800 / 8 = 100 optimizer steps
    'warmup_steps': 10,
    'gradient_accumulation': 8,
    'max_grad_norm': 1.0,
    'eval_every_n_steps': 50,
    'save_every_n_steps': 500,  # don't checkpoint during comparison
    'use_wandb': False,
}

LORA_CONFIG = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)


def _free_gpu():
    """Aggressively free GPU memory between experiments."""
    gc.collect()
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"  GPU: {free / 1e9:.1f} / {total / 1e9:.1f} GB free")


def cognitive_generate(model, tokenizer, input_ids, attention_mask, max_new_tokens=64):
    """Greedy decode through CognitiveModel for configs that need cognitive blocks during eval.

    Only needed when B2 (episodic memory) is active, since it modifies hidden
    states post-transformer. For other configs, base.generate() is faster and
    equivalent.
    """
    model.eval()
    generated = input_ids.clone()
    mask = attention_mask.clone() if attention_mask is not None else torch.ones_like(input_ids)

    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids=generated, attention_mask=mask)
        next_token = outputs["logits"][:, -1, :].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)
        mask = torch.cat([mask, torch.ones_like(next_token)], dim=1)
        if next_token.item() == tokenizer.eos_token_id:
            break

    return generated


def run_experiment(name: str, block_config: dict):
    """Run a single training experiment with the given block config."""
    print(f"\n{'='*60}")
    print(f"  EXPERIMENT: {name}")
    print(f"  Blocks: {[k for k,v in block_config.items() if v]}")
    print(f"{'='*60}\n")

    # Aggressively free GPU memory from previous run
    _free_gpu()

    # Clear cached modules so we get fresh imports
    for mod_name in list(sys.modules):
        if mod_name.startswith('cognitive_llm'):
            del sys.modules[mod_name]
    from cognitive_llm.models.cognitive_model import CognitiveModel
    from cognitive_llm.training.trainer import CognitiveTrainer

    # Reload base model fresh (LoRA can't be re-applied to an already-wrapped model)
    fresh_base = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map='auto',
        torch_dtype=compute_dtype,
    )
    fresh_base = prepare_model_for_kbit_training(fresh_base)
    fresh_base = get_peft_model(fresh_base, LORA_CONFIG)

    # Build cognitive model
    model = CognitiveModel(fresh_base, block_config)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable params: {trainable:,}")

    # Train
    trainer = CognitiveTrainer(
        model=model,
        train_dataloader=train_loader,
        config=TRAINING_CONFIG,
    )
    losses = trainer.train()

    # Evaluate on GSM8K
    # Use cognitive_generate only when B2 is active (modifies hidden states).
    # For other configs, base.generate() is equivalent and uses KV-cache.
    needs_cognitive_gen = block_config.get('use_block2', False)
    test_dataset = load_dataset('gsm8k', 'main', split='test[:20]')
    model.eval()
    correct = 0
    for example in test_dataset:
        prompt = f"Question: {example['question']}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256).to('cuda')
        with torch.no_grad():
            if needs_cognitive_gen:
                generated = cognitive_generate(
                    model, tokenizer,
                    inputs['input_ids'], inputs['attention_mask'],
                    max_new_tokens=64,
                )
            else:
                generated = fresh_base.generate(
                    **inputs, max_new_tokens=64, do_sample=False,
                )
        response = tokenizer.decode(generated[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        pred = re.search(r'####\s*([-\d,\.]+)', response)
        target = re.search(r'####\s*([-\d,\.]+)', example['answer'])
        if pred and target and pred.group(1).replace(',','') == target.group(1).replace(',',''):
            correct += 1

    accuracy = correct / len(test_dataset) * 100
    result = {
        'losses': losses,
        'final_loss': losses[-1],
        'first_10_avg': sum(losses[:10]) / 10,
        'last_10_avg': sum(losses[-10:]) / 10,
        'gsm8k_accuracy': accuracy,
        'trainable_params': trainable,
        'config': block_config,
    }
    all_results[name] = result

    print(f"\n  Final loss: {result['final_loss']:.4f}")
    print(f"  Loss trend: {result['first_10_avg']:.4f} → {result['last_10_avg']:.4f}")
    print(f"  GSM8K accuracy: {accuracy:.1f}%")
    print(f"  Eval method: {'cognitive_generate' if needs_cognitive_gen else 'base.generate'}")

    # Free memory — explicitly delete large objects
    del model, trainer
    del fresh_base
    _free_gpu()

    return result

print("Experiment runner ready. Call run_experiment(name, config) to run.")

In [6]:
# ── Define experiment configs ──
# Each config is a name + block flags. Run them sequentially.

EXPERIMENTS = {
    'LoRA only (baseline)': {
        'use_block1': False, 'use_block2': False, 'use_block3': False,
        'use_block4': False, 'use_block5': False, 'use_block6': False,
    },
    'B6 (HomeostaticNorm)': {
        'use_block1': False, 'use_block2': False, 'use_block3': False,
        'use_block4': False, 'use_block5': False, 'use_block6': True,
    },
    'B1+B6 (Surprise+Homeo)': {
        'use_block1': True,  'use_block2': False, 'use_block3': False,
        'use_block4': False, 'use_block5': False, 'use_block6': True,
    },
    'B1+B2+B6 (Surprise+Memory+Homeo)': {
        'use_block1': True,  'use_block2': True,  'use_block3': False,
        'use_block4': False, 'use_block5': False, 'use_block6': True,
    },
}

print(f"Defined {len(EXPERIMENTS)} experiments:")
for name, cfg in EXPERIMENTS.items():
    active = [k.replace('use_','') for k,v in cfg.items() if v]
    print(f"  {name}: {active or ['none']}")

Defined 4 experiments:
  LoRA only (baseline): ['none']
  B6 (HomeostaticNorm): ['block6']
  B1+B6 (Surprise+Homeo): ['block1', 'block6']
  B1+B2+B6 (Surprise+Memory+Homeo): ['block1', 'block2', 'block6']


In [7]:
# ── Tokenize dataset (shared across experiments) ──
MAX_SEQ_LEN = 256

dataset = load_dataset('gsm8k', 'main', split='train[:128]')

def tokenize_gsm8k(examples):
    texts = [
        f'Question: {q}\nAnswer: {a}'
        for q, a in zip(examples['question'], examples['answer'])
    ]
    tokenized = tokenizer(texts, truncation=True, max_length=MAX_SEQ_LEN, padding='max_length')
    tokenized['labels'] = [ids.copy() for ids in tokenized['input_ids']]
    return tokenized

tokenized_dataset = dataset.map(tokenize_gsm8k, batched=True, remove_columns=dataset.column_names)
tokenized_dataset.set_format(type='torch')

from torch.utils.data import DataLoader
train_loader = DataLoader(tokenized_dataset, batch_size=1, shuffle=True)
print(f"Dataset ready: {len(tokenized_dataset)} samples, seq_len={MAX_SEQ_LEN}")

Dataset ready: 128 samples, seq_len=256


In [8]:
# ── Run all experiments ──
# This takes ~15-20 min on T4 (4 configs x ~4 min each)
# You can also run them one at a time by commenting out lines.

for name, config in EXPERIMENTS.items():
    if name not in all_results:  # skip already-completed experiments
        run_experiment(name, config)
    else:
        print(f"Skipping {name} (already done)")

print(f"\nAll {len(all_results)} experiments complete!")


  EXPERIMENT: LoRA only (baseline)
  Blocks: []



`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1002.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 397.75 MiB is free. Process 15861 has 2.56 GiB memory in use. Process 24153 has 3.79 GiB memory in use. Process 38237 has 3.79 GiB memory in use. Including non-PyTorch memory, this process has 4.03 GiB memory in use. Of the allocated memory 3.69 GiB is allocated by PyTorch, and 222.35 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# ── Comparison: Loss curves ──
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Loss curves
for name, result in all_results.items():
    ax1.plot(result['losses'], label=name, alpha=0.8)
ax1.set_xlabel('Optimizer Step')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss by Configuration')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Bar chart: final metrics
names = list(all_results.keys())
final_losses = [r['last_10_avg'] for r in all_results.values()]
accuracies = [r['gsm8k_accuracy'] for r in all_results.values()]

x = range(len(names))
bars = ax2.bar(x, accuracies, color=['#ccc', '#8cb4d9', '#4a90d9', '#2563a0'])
ax2.set_xticks(x)
ax2.set_xticklabels([n.split(' (')[0] for n in names], rotation=15, ha='right', fontsize=9)
ax2.set_ylabel('GSM8K Accuracy (%)')
ax2.set_title('GSM8K Accuracy by Configuration (20 samples)')
ax2.grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars, accuracies):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{acc:.0f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Summary table ──
print(f"{'Config':<35} {'Params':>10} {'Loss(first10)':>14} {'Loss(last10)':>13} {'GSM8K':>7}")
print("-" * 85)
for name, r in all_results.items():
    print(f"{name:<35} {r['trainable_params']:>10,} {r['first_10_avg']:>14.4f} {r['last_10_avg']:>13.4f} {r['gsm8k_accuracy']:>6.1f}%")

print("\n Key observations:")
if len(all_results) >= 2:
    names = list(all_results.keys())
    baseline_loss = all_results[names[0]]['last_10_avg']
    for name in names[1:]:
        r = all_results[name]
        delta = baseline_loss - r['last_10_avg']
        print(f"  {name}: loss {'↓' if delta > 0 else '↑'}{abs(delta):.4f} vs baseline")